# Step 1: Data Loading

In [ ]:

import os
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, confusion_matrix, f1_score, classification_report
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import Xception
from tensorflow.keras.preprocessing import image
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import seaborn as sns
from staintools import StainNormalizer

# Path to dataset directory
dataset_root = 'path_to_bach_dataset'
img_size = (512, 682)

def load_image_paths(root):
    classes = sorted(os.listdir(root))
    filepaths = []
    labels = []
    for idx, cls in enumerate(classes):
        cls_dir = os.path.join(root, cls)
        for fname in os.listdir(cls_dir):
            if fname.lower().endswith(('png','jpg','jpeg','tif')):
                filepaths.append(os.path.join(cls_dir, fname))
                labels.append(idx)
    return np.array(filepaths), np.array(labels), classes

filepaths, labels, class_names = load_image_paths(dataset_root)


# Step 2: Preprocessing

In [ ]:

def macenko_normalize(img_path, normalizer=None):
    img = image.load_img(img_path)
    img = image.img_to_array(img)
    if normalizer is not None:
        img = normalizer.transform(img)
    img = tf.image.resize(img, img_size)
    return img

reference_image_path = 'path_to_reference_image'
reference_img = image.load_img(reference_image_path)
reference_img = image.img_to_array(reference_img)
normalizer = StainNormalizer(method='macenko')
normalizer.fit(reference_img.astype(np.uint8))


# Step 3: Model Definition

In [ ]:
def build_model():
    
    img_in = layers.Input(shape=(img_size[0], img_size[1], 3))
    base = Xception(weights='imagenet', include_top=False, input_tensor=img_in)
    add_layers = [l.name for l in base.layers if l.name.startswith('add_')]
    layer_names = add_layers[-6:]
    
    gap_features = [layers.GlobalAveragePooling2D()(base.get_layer(name).output) for name in layer_names]
    concat = layers.Concatenate()(gap_features)
    
    dense1 = layers.Dense(512, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01))(concat)
    drop1 = layers.Dropout(0.1)(dense1)
    dense2 = layers.Dense(512, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01))(drop1)
    drop2 = layers.Dropout(0.1)(dense2)
    output = layers.Dense(len(class_names), activation='softmax')(drop2)
    model = Model(inputs=img_in, outputs=output)
    model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(1e-5), metrics=['accuracy'])
    return model


# Step 4: Cross-validation Setup

# Step 5: Training & Evaluation

In [ ]:

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_metrics = []
histories = []

for fold, (train_idx, val_idx) in enumerate(skf.split(filepaths, labels), 1):
    model = build_model()
    print(f'Fold {fold}')
    train_paths, val_paths = filepaths[train_idx], filepaths[val_idx]
    y_train, y_val = labels[train_idx], labels[val_idx]

    def preprocess(paths, labels, augment=False):
        imgs = []
        for p in paths:
            img = macenko_normalize(p, normalizer)
            if augment:
                img = tf.image.random_flip_left_right(img)
                img = tf.image.random_flip_up_down(img)
            imgs.append(img)
        imgs = tf.stack(imgs)
        labels_cat = to_categorical(labels, num_classes=len(class_names))
        return imgs, labels_cat

    x_train, y_train_cat = preprocess(train_paths, y_train, augment=True)
    x_val, y_val_cat = preprocess(val_paths, y_val)

    checkpoint = tf.keras.callbacks.ModelCheckpoint(f'xception_fold{fold}.h5',
                                                    monitor='val_accuracy',
                                                    save_best_only=True)
    history = model.fit(x_train, y_train_cat, epochs=3,
                        validation_data=(x_val, y_val_cat),
                        callbacks=[checkpoint])
    histories.append(history.history)

    preds = model.predict(x_val)
    y_true = y_val
    y_score = preds
    y_pred = preds.argmax(axis=1)

    f1_micro = f1_score(y_true, y_pred, average='micro')
    f1_macro = f1_score(y_true, y_pred, average='macro')
    pr_precision, pr_recall, _ = precision_recall_curve(to_categorical(y_true, num_classes=len(class_names)).ravel(),
                                                       y_score.ravel())
    pr_auc = auc(pr_recall, pr_precision)
    roc_auc = roc_auc_score(to_categorical(y_true, num_classes=len(class_names)), y_score, multi_class='ovr')
    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred, target_names=class_names)

    # plot confusion matrix
    plt.figure()
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'Confusion Matrix Fold {fold}')
    plt.show()
    # plot training curves
    plt.figure()
    plt.plot(history.history['accuracy'], label='train_acc')
    plt.plot(history.history['val_accuracy'], label='val_acc')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.title(f'Accuracy Fold {fold}')
    plt.show()
    plt.figure()
    plt.plot(history.history['loss'], label='train_loss')
    plt.plot(history.history['val_loss'], label='val_loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title(f'Loss Fold {fold}')
    plt.show()

    fold_metrics = dict(f1_micro=f1_micro, f1_macro=f1_macro,
                        pr_auc=pr_auc, roc_auc=roc_auc,
                        cm=cm, report=report)
    all_metrics.append(fold_metrics)


# Step 6: Visualization & Summary

In [ ]:
for i, m in enumerate(all_metrics, 1):
    print(f'Fold {i}:')
    print('F1 micro:', m['f1_micro'])
    print('F1 macro:', m['f1_macro'])
    print('PR AUC:', m['pr_auc'])
    print('ROC AUC:', m['roc_auc'])
    print('Classification Report:')
    print(m['report'])

avg_f1_micro = np.mean([m['f1_micro'] for m in all_metrics])
avg_f1_macro = np.mean([m['f1_macro'] for m in all_metrics])
avg_pr_auc = np.mean([m['pr_auc'] for m in all_metrics])
avg_roc_auc = np.mean([m['roc_auc'] for m in all_metrics])

print('
Average Metrics:')
print('F1 micro:', avg_f1_micro)
print('F1 macro:', avg_f1_macro)
print('PR AUC:', avg_pr_auc)
print('ROC AUC:', avg_roc_auc)
